# 🧠 File 03 — Xây dựng mô hình (Model Building)
**Brain Tumor Classification MRI Dataset**

File này sẽ xây dựng **2 mô hình** để so sánh:
- **Model 1:** CNN tự xây (Baseline) — đơn giản, hiểu rõ từng bước
- **Model 2:** Transfer Learning với EfficientNetB0 — mạnh hơn, chính xác hơn

Sau đó so sánh kết quả để chọn model tốt nhất.

## 📦 Phần A — Import thư viện

In [ ]:
import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    ModelCheckpoint,    # Lưu model tốt nhất
    EarlyStopping,      # Dừng sớm khi không cải thiện
    ReduceLROnPlateau   # Giảm learning rate khi bị tắc
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from pathlib import Path

# Kiểm tra GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'✅ TensorFlow: {tf.__version__}')
print(f'   GPU available: {len(gpus) > 0}')
if gpus:
    print(f'   GPU: {gpus[0].name}')
else:
    print('   Đang dùng CPU — train sẽ chậm hơn, hãy kiên nhẫn!')

## ⚙️ Phần B — Load lại cấu hình và dữ liệu từ file 02

In [ ]:
# ============================================================
# CONFIG - must match 02_Preprocessing_v2.ipynb
# ============================================================
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = Path(os.environ.get('MODEL_OUTPUT_DIR', PROJECT_ROOT))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR  = PROJECT_ROOT / 'data'
TRAIN_DIR = DATA_DIR / 'Training'
TEST_DIR  = DATA_DIR / 'Testing'
CSV_DIR   = PROJECT_ROOT

IMG_SIZE    = (240, 240)
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2
RANDOM_SEED = 42
NUM_CLASSES = 4
EPOCHS_CNN       = 25
EPOCHS_TL_STAGE1 = 12
EPOCHS_TL_STAGE2A = 8
EPOCHS_TL_STAGE2B = 10
GLIOMA_WEIGHT_MULTIPLIER = 1.20

CLASSES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
CLASS_LABELS = {
    'glioma_tumor':     'Glioma',
    'meningioma_tumor': 'Meningioma',
    'no_tumor':         'Khong co u',
    'pituitary_tumor':  'Pituitary'
}

assert TRAIN_DIR.exists(), f'Missing training folder: {TRAIN_DIR}'
assert TEST_DIR.exists(),  f'Missing testing folder: {TEST_DIR}'


def build_dataframe(base_dir, classes):
    records = []
    for cls in classes:
        cls_path = base_dir / cls
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for img_path in cls_path.glob(ext):
                records.append({'filepath': str(img_path.resolve()), 'label': cls})
    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f'No images found under {base_dir}')
    return df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


def csvs_are_valid(*dfs):
    for df in dfs:
        if not {'filepath', 'label'}.issubset(df.columns):
            return False
        if set(df['label'].unique()) - set(CLASSES):
            return False
        if not df['filepath'].map(lambda p: Path(p).exists()).all():
            return False
    return True

csv_paths = [CSV_DIR / 'df_train.csv', CSV_DIR / 'df_val.csv', CSV_DIR / 'df_test.csv']
if all(p.exists() for p in csv_paths):
    df_train = pd.read_csv(csv_paths[0])
    df_val   = pd.read_csv(csv_paths[1])
    df_test  = pd.read_csv(csv_paths[2])
else:
    df_train = df_val = df_test = None

if df_train is None or not csvs_are_valid(df_train, df_val, df_test):
    print('CSV files missing/invalid. Rebuilding split from ./data ...')
    df_train_all = build_dataframe(TRAIN_DIR, CLASSES)
    df_test      = build_dataframe(TEST_DIR, CLASSES)
    df_train, df_val = train_test_split(
        df_train_all,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=df_train_all['label']
    )
    df_train = df_train.reset_index(drop=True)
    df_val   = df_val.reset_index(drop=True)
    df_train.to_csv(csv_paths[0], index=False)
    df_val.to_csv(csv_paths[1], index=False)
    df_test.to_csv(csv_paths[2], index=False)

print(f'Model output directory: {OUTPUT_DIR}')
print('Loaded data split:')
print(f'   Training   : {len(df_train)} images')
print(f'   Validation : {len(df_val)} images')
print(f'   Testing    : {len(df_test)} images')
print('Class distribution train:', df_train['label'].value_counts().reindex(CLASSES).to_dict())
print('Class distribution val  :', df_val['label'].value_counts().reindex(CLASSES).to_dict())
print('Class distribution test :', df_test['label'].value_counts().reindex(CLASSES).to_dict())

In [ ]:
# ============================================================
# Data generators - synchronized with 02_Preprocessing_v2.ipynb
# ============================================================
from sklearn.utils import compute_class_weight

train_datagen = ImageDataGenerator(
    rotation_range=7,
    width_shift_range=0.03,
    height_shift_range=0.03,
    shear_range=0.0,
    zoom_range=0.07,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.95, 1.05],
    fill_mode='constant',
    cval=0
)
val_test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=True, seed=RANDOM_SEED
)
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=False
)
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_test, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=False
)

assert val_generator.class_indices == train_generator.class_indices
assert test_generator.class_indices == train_generator.class_indices

CLASS_INDICES  = train_generator.class_indices
INDEX_TO_CLASS = {v: k for k, v in CLASS_INDICES.items()}
print(f'Class mapping: {CLASS_INDICES}')
print(f'Samples: train={train_generator.samples}, val={val_generator.samples}, test={test_generator.samples}')

labels = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))

# CNN is highly precise but under-recalls Glioma, so use a small cost-sensitive adjustment.
cnn_class_weight_dict = class_weight_dict.copy()
glioma_idx = CLASS_INDICES['glioma_tumor']
cnn_class_weight_dict[glioma_idx] *= GLIOMA_WEIGHT_MULTIPLIER

print('\nFrequency-balanced weights:')
for idx, cls in enumerate(CLASSES):
    print(f'   {CLASS_LABELS[cls]:<15}: {class_weight_dict[idx]:.3f}')
print('\nCNN tuned weights:')
for idx, cls in enumerate(CLASSES):
    print(f'   {CLASS_LABELS[cls]:<15}: {cnn_class_weight_dict[idx]:.3f}')


def training_metrics():
    return [
        keras.metrics.CategoricalAccuracy(name='accuracy'),
        keras.metrics.F1Score(average='macro', name='macro_f1')
    ]


## 🏗️ Phần C — Model 1: CNN tự xây (Baseline)

Xây một mạng CNN từ đầu để hiểu cách hoạt động:
- **Conv2D + MaxPooling**: trích xuất đặc trưng (cạnh, texture, hình dạng)
- **BatchNormalization**: ổn định quá trình train
- **Dropout**: giảm overfitting
- **Dense**: phân loại cuối cùng

In [ ]:
# ============================================================
# Xây CNN Baseline
# ============================================================
def build_cnn_baseline(input_shape, num_classes):
    """
    CNN tự xây với 4 khối Conv.
    input_shape: (224, 224, 3)
    num_classes: 4
    """
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=input_shape),

        # ── Khối 1: Học đặc trưng cơ bản (cạnh, góc) ──
        layers.Conv2D(32, (3,3), activation='relu',
                      padding='same'),
        layers.BatchNormalization(),   # Chuẩn hoá để train ổn định hơn
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),     # Giảm kích thước ảnh xuống 1/2
        layers.Dropout(0.25),          # Bỏ 25% neuron ngẫu nhiên → giảm overfitting

        # ── Khối 2: Học đặc trưng phức tạp hơn ──
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Khối 3: Học đặc trưng trừu tượng hơn ──
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Khối 4: Học đặc trưng cao cấp ──
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.3),

        # ── Phân loại ──
        layers.GlobalAveragePooling2D(), # Thay Flatten — giảm tham số, tránh overfit
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),             # Dropout mạnh hơn ở lớp fully connected
        layers.Dense(num_classes, activation='softmax')  # Softmax cho 4 class
    ], name='CNN_Baseline')

    return model

cnn_model = build_cnn_baseline((IMG_SIZE[0], IMG_SIZE[1], 3), NUM_CLASSES)

# Compile: chọn optimizer, loss function, metric
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',  # Loss cho multi-class classification
    metrics=training_metrics()
)

cnn_model.summary()

In [ ]:
# ============================================================
# CNN tuned callbacks - preserve the old best_cnn_model.h5
# ============================================================
callbacks_cnn = [
    ModelCheckpoint(
        str(OUTPUT_DIR / 'best_cnn_tuned_model.h5'),
        monitor='val_macro_f1', mode='max',
        save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_macro_f1', mode='max',
        patience=6, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', mode='min',
        factor=0.5, patience=3, min_lr=1e-7, verbose=1
    )
]
print('CNN tuned checkpoint: best_cnn_tuned_model.h5')
print('Selection metric: validation macro F1')


In [ ]:
# ============================================================
# Build + train tuned CNN from scratch
# ============================================================
keras.utils.set_random_seed(RANDOM_SEED)
cnn_model = build_cnn_baseline((IMG_SIZE[0], IMG_SIZE[1], 3), NUM_CLASSES)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=training_metrics()
)
cnn_model.summary()

print('\nTraining tuned CNN with a 20% Glioma weight increase...')
history_cnn = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    callbacks=callbacks_cnn,
    class_weight=cnn_class_weight_dict,
    verbose=2
)
print('\nTuned CNN training complete.')


## 📈 Phần D — Vẽ đồ thị quá trình train CNN

Xem accuracy và loss qua từng epoch để phát hiện:
- **Underfitting**: cả train và val đều thấp
- **Overfitting**: train cao nhưng val thấp/giảm

In [ ]:
# ============================================================
# Hàm vẽ đồ thị training history — dùng lại cho cả 2 model
# ============================================================
def plot_training_history(history, model_name, save_path=None):
    """
    Vẽ đồ thị Accuracy và Loss theo epoch.
    history   : kết quả trả về từ model.fit()
    model_name: tên model để đặt tiêu đề
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Quá trình Training — {model_name}',
                 fontsize=14, fontweight='bold')

    epochs_ran = range(1, len(history.history['accuracy']) + 1)

    # --- Accuracy ---
    axes[0].plot(epochs_ran, history.history['accuracy'],
                 'b-o', markersize=4, label='Train Accuracy')
    axes[0].plot(epochs_ran, history.history['val_accuracy'],
                 'r-o', markersize=4, label='Val Accuracy')
    axes[0].set_title('Accuracy theo Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].set_ylim([0, 1])
    # Đánh dấu epoch tốt nhất
    best_epoch = np.argmax(history.history['val_accuracy'])
    axes[0].axvline(x=best_epoch+1, color='green', linestyle='--',
                    alpha=0.7, label=f'Best epoch: {best_epoch+1}')
    axes[0].legend()

    # --- Loss ---
    axes[1].plot(epochs_ran, history.history['loss'],
                 'b-o', markersize=4, label='Train Loss')
    axes[1].plot(epochs_ran, history.history['val_loss'],
                 'r-o', markersize=4, label='Val Loss')
    axes[1].set_title('Loss theo Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'💾 Đã lưu: {save_path}')
    plt.show()

    # In kết quả tốt nhất
    best_val_acc = max(history.history['val_accuracy'])
    print(f'\n🏆 Val Accuracy tốt nhất: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')


plot_training_history(history_cnn, 'CNN Baseline', str(OUTPUT_DIR / 'model_cnn_tuned_history.png'))

## 🚀 Phần E — Model 2: Transfer Learning (EfficientNetB0)

**Transfer Learning** là dùng lại một mạng đã được train trên triệu ảnh (ImageNet),
rồi fine-tune cho bài toán của mình.

Cách tiếp cận:
1. **Giai đoạn 1 (Feature Extraction):** Đóng băng EfficientNet, chỉ train lớp đầu ra mới
2. **Giai đoạn 2 (Fine-tuning):** Mở băng một số lớp cuối, train tiếp với LR nhỏ hơn

In [ ]:
# ============================================================
# EfficientNetB0 with a compact classification head
# ============================================================
def build_transfer_model(input_shape, num_classes):
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)  # Keep BatchNorm in inference mode.
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB0_Tuned')
    return model, base_model


keras.utils.set_random_seed(RANDOM_SEED)
tl_model, base_model = build_transfer_model((IMG_SIZE[0], IMG_SIZE[1], 3), NUM_CLASSES)
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss='categorical_crossentropy',
    metrics=training_metrics()
)

print(f'Total params       : {tl_model.count_params():,}')
print(f'Trainable params   : {sum(np.prod(v.shape) for v in tl_model.trainable_variables):,}')
print(f'Non-trainable params: {sum(np.prod(v.shape) for v in tl_model.non_trainable_variables):,}')


In [ ]:
# ============================================================
# Stage 1: feature extraction (frozen EfficientNet backbone)
# ============================================================
callbacks_tl_stage1 = [
    ModelCheckpoint(str(OUTPUT_DIR / 'best_tl_stage1_model.h5'), monitor='val_macro_f1', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_macro_f1', mode='max', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.5,
                      patience=2, min_lr=1e-6, verbose=1)
]

print('Stage 1: frozen backbone, LR=3e-4')
history_tl_s1 = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL_STAGE1,
    validation_data=val_generator,
    callbacks=callbacks_tl_stage1,
    class_weight=class_weight_dict,
    verbose=2
)
print('Stage 1 complete')


In [ ]:
# ============================================================
# Stage 2: progressive fine-tuning with separate checkpoints
# ============================================================
base_model.trainable = True

# Stage 2a: unfreeze last 20 non-BN layers.
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.03),
    metrics=training_metrics()
)
callbacks_tl_stage2a = [
    ModelCheckpoint(str(OUTPUT_DIR / 'best_tl_stage2a_model.h5'), monitor='val_macro_f1', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_macro_f1', mode='max', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.5,
                      patience=2, min_lr=5e-8, verbose=1)
]
print('Stage 2a: last 20 non-BN layers, LR=5e-6')
history_tl_s2a = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL_STAGE2A,
    validation_data=val_generator,
    callbacks=callbacks_tl_stage2a,
    class_weight=class_weight_dict,
    verbose=2
)

# Stage 2b: unfreeze last 80 non-BN layers, with a lower LR.
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-80:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-6),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.03),
    metrics=training_metrics()
)
callbacks_tl_stage2b = [
    ModelCheckpoint(str(OUTPUT_DIR / 'best_tl_stage2b_model.h5'), monitor='val_macro_f1', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_macro_f1', mode='max', patience=4,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.5,
                      patience=2, min_lr=1e-8, verbose=1)
]
print('Stage 2b: last 80 non-BN layers, LR=1e-6')
history_tl_s2b = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL_STAGE2B,
    validation_data=val_generator,
    callbacks=callbacks_tl_stage2b,
    class_weight=class_weight_dict,
    verbose=2
)
print('Fine-tuning complete')


In [ ]:
# ============================================================
# Ghép 3 giai đoạn history để vẽ đồ thị liền mạch
# ============================================================
class MergedHistory:
    """Ghép 2 history objects thành 1 để vẽ cùng nhau."""
    def __init__(self, h1, h2):
        self.history = {
            key: h1.history[key] + h2.history[key]
            for key in h1.history
        }

# Ghép cả 3 giai đoạn: Stage1 + Stage2a + Stage2b
history_tl = MergedHistory(MergedHistory(history_tl_s1, history_tl_s2a), history_tl_s2b)
plot_training_history(history_tl, 'EfficientNetB0 Tuned (Progressive Unfreeze)', str(OUTPUT_DIR / 'model_tl_tuned_history.png'))


## 📊 Phần F — So sánh 2 Model trên Validation Set

In [ ]:
# ============================================================
# Select global best EfficientNet stage on validation macro F1
# ============================================================
def predict_metrics(model, generator):
    generator.reset()
    y_prob = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = generator.classes
    y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=NUM_CLASSES)
    loss = keras.losses.CategoricalCrossentropy()(y_true_onehot, y_prob).numpy()
    acc = np.mean(y_pred == y_true)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return float(loss), float(acc), float(macro_f1)

candidate_paths = [
    str(OUTPUT_DIR / 'best_tl_stage1_model.h5'),
    str(OUTPUT_DIR / 'best_tl_stage2a_model.h5'),
    str(OUTPUT_DIR / 'best_tl_stage2b_model.h5'),
]
stage_metrics = {}
for candidate_path in candidate_paths:
    candidate = keras.models.load_model(candidate_path, compile=False)
    loss, acc, macro_f1 = predict_metrics(candidate, val_generator)
    stage_metrics[candidate_path] = {
        'val_loss': loss,
        'val_accuracy': acc,
        'val_macro_f1': macro_f1,
    }
    print(f'{candidate_path}: val_acc={acc:.4f}, val_macro_f1={macro_f1:.4f}, val_loss={loss:.4f}')

best_tl_path = max(stage_metrics, key=lambda p: stage_metrics[p]['val_macro_f1'])
shutil.copyfile(best_tl_path, str(OUTPUT_DIR / 'best_tl_tuned_model.h5'))
(OUTPUT_DIR / 'tl_stage_validation_metrics.json').write_text(json.dumps(stage_metrics, indent=2), encoding='utf-8')
print(f'Global best TL stage: {best_tl_path}')
print('Saved as: best_tl_tuned_model.h5')

best_cnn = keras.models.load_model(str(OUTPUT_DIR / 'best_cnn_tuned_model.h5'), compile=False)
best_tl = keras.models.load_model(str(OUTPUT_DIR / 'best_tl_tuned_model.h5'), compile=False)
cnn_val_loss, cnn_val_acc, cnn_val_f1 = predict_metrics(best_cnn, val_generator)
tl_val_loss, tl_val_acc, tl_val_f1 = predict_metrics(best_tl, val_generator)

print('\nVALIDATION COMPARISON')
print(f'CNN tuned       : Acc={cnn_val_acc:.4f}, Macro-F1={cnn_val_f1:.4f}, Loss={cnn_val_loss:.4f}')
print(f'EfficientNet TL : Acc={tl_val_acc:.4f}, Macro-F1={tl_val_f1:.4f}, Loss={tl_val_loss:.4f}')

if tl_val_f1 > cnn_val_f1:
    BEST_MODEL, BEST_MODEL_NAME = best_tl, 'EfficientNetB0 Tuned'
else:
    BEST_MODEL, BEST_MODEL_NAME = best_cnn, 'CNN Tuned'
print(f'Selected for evaluation: {BEST_MODEL_NAME}')


In [ ]:
# ============================================================
# Validation comparison plot
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Tuned Model Validation Comparison', fontsize=14, fontweight='bold')
model_names = ['CNN Tuned', 'EfficientNetB0 Tuned']
values = [
    ('Accuracy (%)', [cnn_val_acc * 100, tl_val_acc * 100]),
    ('Macro F1 (%)', [cnn_val_f1 * 100, tl_val_f1 * 100]),
    ('Loss', [cnn_val_loss, tl_val_loss]),
]
colors = ['#2196F3', '#4CAF50']
for ax, (title, metric_values) in zip(axes, values):
    ax.bar(model_names, metric_values, color=colors)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=10)
    for i, value in enumerate(metric_values):
        ax.text(i, value + max(metric_values) * 0.02, f'{value:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_tuned_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_tuned_comparison.png')


## Phan G - Final test evaluation for tuned checkpoints


In [ ]:
# ============================================================
# Final test evaluation for tuned checkpoints
# ============================================================
def evaluate_test_model(name, model_path):
    model = keras.models.load_model(model_path, compile=False)
    test_generator.reset()
    y_prob = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_generator.classes
    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    accuracy = float(np.mean(y_true == y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    print(f'\n{name}: accuracy={accuracy:.4f}, macro_precision={precision:.4f}, '
          f'macro_recall={recall:.4f}, macro_f1={macro_f1:.4f}')
    print(classification_report(
        y_true, y_pred,
        target_names=[CLASS_LABELS[c] for c in CLASSES],
        digits=4, zero_division=0
    ))
    return {
        'accuracy': accuracy,
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(macro_f1),
        'confusion_matrix': cm.tolist(),
    }

final_results = {
    'CNN Tuned': evaluate_test_model('CNN Tuned', str(OUTPUT_DIR / 'best_cnn_tuned_model.h5')),
    'EfficientNetB0 Tuned': evaluate_test_model('EfficientNetB0 Tuned', str(OUTPUT_DIR / 'best_tl_tuned_model.h5')),
}
(OUTPUT_DIR / 'model_tuned_test_results.json').write_text(json.dumps(final_results, indent=2), encoding='utf-8')
print('Saved: model_tuned_test_results.json')

import seaborn as sns
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle(f'Tuned Models - Test Confusion Matrices ({test_generator.samples} images)',
             fontsize=16, fontweight='bold')
for col, (name, metrics) in enumerate(final_results.items()):
    cm = np.array(metrics['confusion_matrix'])
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    labels_display = [CLASS_LABELS[c] for c in CLASSES]
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=labels_display, yticklabels=labels_display, ax=axes[0, col])
    axes[0, col].set_title(f'{name} - Counts\nAcc={metrics["accuracy"]*100:.2f}%, F1={metrics["macro_f1"]*100:.2f}%')
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Greens', vmin=0, vmax=100, cbar=False,
                xticklabels=labels_display, yticklabels=labels_display, ax=axes[1, col])
    axes[1, col].set_title(f'{name} - Row-normalized recall (%)')
    for row in range(2):
        axes[row, col].set_xlabel('Predicted')
        axes[row, col].set_ylabel('True')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'model_tuned_confusion_matrices.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved: model_tuned_confusion_matrices.png')
